# 03. Matriz Final para Clustering

## Goal
Transform the engineered company features into the final matrix to be consumed by the clustering stage.


## Inputs
- `outputs/features_empresas.parquet`

## Outputs
- `outputs/matriz_clustering.parquet`
- `outputs/matriz_clustering.csv`
- Optional data dictionary for the final model inputs.


## Suggested Final Steps
- Remove leakage columns and pure identifiers.
- Decide how to impute missing values.
- Encode categorical variables if they remain.
- Scale numeric features if the chosen clustering method needs it.
- Keep a versioned export of the final matrix.


In [ ]:

# ── Helpers y rutas ──────────────────────────────────────────────────────────
from pathlib import Path
import pickle
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "01_data_ingestion_enrichment").is_dir() and (p / "02_data_cleaning").is_dir():
            return p
    raise FileNotFoundError("No se pudo localizar la raíz.")

def ensure_dir(d): Path(d).mkdir(parents=True, exist_ok=True); return Path(d)

def save_df_csv(df, path, *, index=False, encoding="utf-8-sig"):
    path = Path(path); ensure_dir(path.parent)
    df.to_csv(path, index=index, encoding=encoding)
    print(f"[OK] Guardado: {path.resolve()}  shape: {df.shape}"); return path

ROOT       = find_project_root()
OUTPUT_DIR = ROOT / "03_feature_engineering" / "outputs"
ensure_dir(OUTPUT_DIR)

print(f"[CONFIG] ROOT      : {ROOT}")
print(f"[CONFIG] OUTPUT_DIR: {OUTPUT_DIR}")

# ── Cargar features ───────────────────────────────────────────────────────────
_feat_pkl = OUTPUT_DIR / "features_empresas.pkl"
_feat_csv = OUTPUT_DIR / "features_empresas.csv"
if _feat_pkl.exists():
    with open(_feat_pkl, "rb") as f: feat = pickle.load(f)
    print(f"[FEATURES] cargado desde pkl  {feat.shape}")
elif _feat_csv.exists():
    feat = pd.read_csv(_feat_csv)
    print(f"[FEATURES] cargado desde csv  {feat.shape}")
else:
    raise FileNotFoundError(f"No existe features_empresas en {OUTPUT_DIR}")

# ── Columnas excluidas (identificadores / leakage) ────────────────────────────
COLS_EXCLUIR = ["entity_id", "name_norm", "canonical_name",
                "RUC", "verdict", "source_winner", "name_raw"]
cols_excluidas = [c for c in COLS_EXCLUIR if c in feat.columns]

# ── Columnas modelo ───────────────────────────────────────────────────────────
cols_modelo = [c for c in feat.columns if c not in cols_excluidas]
df_model = feat[cols_modelo].copy()
bool_cols = df_model.select_dtypes(include="bool").columns
df_model[bool_cols] = df_model[bool_cols].astype(int)
df_model = df_model.select_dtypes(include=[np.number])
cols_modelo = df_model.columns.tolist()

# ── Resumen de nulos ──────────────────────────────────────────────────────────
resumen_nulos = pd.DataFrame({
    "columna": df_model.columns,
    "n_null":  [int(df_model[c].isna().sum()) for c in df_model.columns],
    "pct_null":[round(df_model[c].isna().mean()*100, 2) for c in df_model.columns],
}).sort_values("pct_null", ascending=False).reset_index(drop=True)
print(f"\n[NULOS]\n{resumen_nulos.to_string()}")

# ── Imputación de nulos con mediana ──────────────────────────────────────────
df_model = df_model.fillna(df_model.median(numeric_only=True))

# ── Escalado (MinMax para clustering basado en distancia) ────────────────────
try:
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()
    arr_scaled = scaler.fit_transform(df_model)
    matriz_clustering = pd.DataFrame(arr_scaled, columns=cols_modelo)
    print("[SCALE] MinMaxScaler aplicado.")
except ImportError:
    print("[SCALE] sklearn no disponible; escalado manual min-max.")
    df_min = df_model.min(); df_max = df_model.max()
    rng = (df_max - df_min).replace(0, 1)
    matriz_clustering = (df_model - df_min) / rng

if "entity_id" in feat.columns:
    matriz_clustering.insert(0, "entity_id", feat["entity_id"].values)

print(f"\n[MATRIZ] shape: {matriz_clustering.shape}")
print(f"[COLS_MODELO]   {cols_modelo}")
print(f"[COLS_EXCLUIDAS]{cols_excluidas}")

# ── Exports ───────────────────────────────────────────────────────────────────
_ = save_df_csv(matriz_clustering, OUTPUT_DIR / "matriz_clustering.csv")
with open(OUTPUT_DIR / "matriz_clustering.pkl", "wb") as f: pickle.dump(matriz_clustering, f)
print("[OK] matriz_clustering.pkl guardado")

_ = save_df_csv(resumen_nulos, OUTPUT_DIR / "resumen_nulos_matriz.csv")
pd.DataFrame({"columna": cols_modelo}).to_csv(OUTPUT_DIR / "columnas_modelo.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"columna": cols_excluidas}).to_csv(OUTPUT_DIR / "columnas_excluidas.csv", index=False, encoding="utf-8-sig")

# ── Verificación ──────────────────────────────────────────────────────────────
for p in [OUTPUT_DIR/"matriz_clustering.csv", OUTPUT_DIR/"matriz_clustering.pkl"]:
    if not p.exists(): raise FileNotFoundError(f"Output faltante: {p}")
    if p.suffix == ".pkl":
        with open(p,"rb") as f: df_chk = pickle.load(f)
    else:
        df_chk = pd.read_csv(p)
    print(f"[OK] {p.name}  shape={df_chk.shape}")
print("\n✓ Notebook 03_matriz_final_clustering completado correctamente.")
display(matriz_clustering.head(10))
